<a href="https://colab.research.google.com/github/mayajdias/ds2002-fa26/blob/main/notebooks/01-foundations/2026_09_25_cleaning_gauntlet_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [ ]:
# Save the original data for later comparisons
raw_df = df.copy()

# Count duplicate rows
num_duplicates = int(df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Record the decision
log('duplicates', 'Dropped exact duplicate rows', num_duplicates)

print('Rows before cleaning:', len(raw_df))
print('Rows after dropping duplicates:', len(df))
print('Duplicates remaining:', df.duplicated().sum())

[duplicates] Dropped exact duplicate rows (15 row(s))
Rows before cleaning: 315
Rows after dropping duplicates: 300
Duplicates remaining: 0


### TODO 2 — clean `price` -> float

In [ ]:
# Count prices containing dollar signs
dollar_sign_rows = int(
    df['price'].astype(str).str.contains('$', regex=False).sum()
)

# Remove dollar signs and convert prices to float
df['price'] = pd.to_numeric(
    df['price'].astype(str)
      .str.replace('$', '', regex=False)
      .str.strip(),
    errors='raise'
).astype(float)

# Record the decision
log('price', 'Removed dollar signs and converted price to float', dollar_sign_rows)

print('Price dtype:', df['price'].dtype)
print('Unique prices:', df['price'].drop_duplicates().sort_values().tolist())

[price] Removed dollar signs and converted price to float (124 row(s))
Price dtype: float64
Unique prices: [6.0, 7.5, 12.0, 24.0]


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [ ]:
# Convert quantity to numeric
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

# Count problematic quantities
missing_qty = int(df['qty'].isna().sum())
negative_qty = int(df['qty'].lt(0).sum())

# Keep only valid positive quantities
df = df.loc[
    df['qty'].notna() & df['qty'].gt(0)
].copy()

# Record the decision
log(
    'qty',
    f'Dropped {missing_qty} missing and {negative_qty} negative quantities',
    missing_qty + negative_qty
)

print('Missing quantities removed:', missing_qty)
print('Negative quantities removed:', negative_qty)
print('Rows after quantity cleaning:', len(df))
print('Minimum remaining quantity:', df['qty'].min())

[qty] Dropped 12 missing and 13 negative quantities (25 row(s))
Missing quantities removed: 12
Negative quantities removed: 13
Rows after quantity cleaning: 275
Minimum remaining quantity: 1.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [ ]:
# Examine the original item names
print('Before cleaning:')
print(df['item'].value_counts())

# Create a mapping from the original names to canonical names
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

# Save original values for comparison
original_item = df['item'].copy()

# Apply the mapping
df['item'] = df['item'].map(ITEM_MAP)

# Verify nothing was left unmapped
assert df['item'].notna().all()

# Count changed names
item_changes = int((original_item != df['item']).sum())

# Record the decision
log(
    'item',
    'Collapsed six spellings into three canonical product names',
    item_changes
)

print('\nAfter cleaning:')
print(df['item'].value_counts())

print('Distinct item names:', df['item'].nunique())

Before cleaning:
item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] Collapsed six spellings into three canonical product names (126 row(s))

After cleaning:
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
Distinct item names: 3


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [ ]:
# Examine the original categories
print('Before cleaning:')
print(df['category'].value_counts())

# Create the category mapping
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

# Save original values
original_category = df['category'].copy()

# Apply the mapping
df['category'] = df['category'].map(CATEGORY_MAP)

# Verify every category was mapped
assert df['category'].notna().all()

# Count the changed categories
category_changes = int(
    (original_category != df['category']).sum()
)

# Record the decision
log(
    'category',
    'Standardized labels and grouped Apparel into Merch',
    category_changes
)

print('\nAfter cleaning:')
print(df['category'].value_counts())

print('Distinct category names:', df['category'].nunique())

Before cleaning:
category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] Standardized labels and grouped Apparel into Merch (132 row(s))

After cleaning:
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64
Distinct category names: 3


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [ ]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {
    'Cheeseburger',
    'Foam Finger',
    'Rain Poncho'
}
assert set(df['category'].unique()) == {
    'Food',
    'Merch',
    'RainGear'
}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [ ]:
# Calculate revenue for every order
df['revenue'] = df['qty'] * df['price']

# Calculate revenue by category, highest first
by_category = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
)

# Print category revenue
print('Revenue by category:')

for category, amount in by_category.items():
    print(f'{category}: ${amount:,.2f}')

# Print total revenue
total_revenue = df['revenue'].sum()

print(f'\nOverall revenue: ${total_revenue:,.2f}')

# Calculate original revenue for the final write-up
raw_prices = pd.to_numeric(
    raw_df['price'].astype(str)
      .str.replace('$', '', regex=False)
      .str.strip()
)

raw_revenue = (raw_df['qty'] * raw_prices).sum()

duplicate_revenue = (
    raw_df.loc[raw_df.duplicated(), 'qty']
    * raw_prices.loc[raw_df.duplicated()]
).sum()

post_duplicate_revenue = raw_revenue - duplicate_revenue

print('\nRevenue comparison for write-up:')
print(f'Before duplicate removal: ${raw_revenue:,.2f}')
print(f'After duplicate removal: ${post_duplicate_revenue:,.2f}')
print(f'Duplicate impact: ${duplicate_revenue:,.2f}')

print(f'\nRevenue if negative quantities were retained: ${post_duplicate_revenue:,.2f}')
print(f'Impact of removing negative quantities: ${total_revenue - post_duplicate_revenue:,.2f}')

Revenue by category:
Food: $1,656.00
Merch: $1,572.00
RainGear: $1,512.00

Overall revenue: $4,740.00

Revenue comparison for write-up:
Before duplicate removal: $4,852.50
After duplicate removal: $4,594.50
Duplicate impact: $258.00

Revenue if negative quantities were retained: $4,594.50
Impact of removing negative quantities: $145.50


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [ ]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Dropped exact duplicate rows,15
1,price,Removed dollar signs and converted price to float,124
2,qty,Dropped 12 missing and 13 negative quantities,25
3,item,Collapsed six spellings into three canonical p...,126
4,category,Standardized labels and grouped Apparel into M...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** Removing duplicate rows had the largest effect on total revenue. Before removing duplicates, total revenue was \$4,852.50. After removing the 15 duplicate rows, revenue decreased to \$4,594.50, a difference of \$258.00. This was the largest revenue change because the duplicate orders were being counted more than once. Removing them prevents revenue from being overstated.

**b)** One decision that someone could reasonably have made differently was how to handle negative quantities. I chose to remove the 13 rows with negative quantities because the assignment asks us to exclude them, allowing the final revenue total to reflect positive-quantity sales. Another reasonable approach would be to retain the negative quantities as refunds, since they represent money being returned to customers. If I had kept those rows, reported revenue would have been \$4,594.50 instead of \$4,740.00, a decrease of \$145.50. I chose to exclude them for this analysis but would recommend tracking refunds separately in a real business report so they are not lost.